#### Ejemplo cálculo diferencial

In [1]:
import sys
sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO

def read_db(datafolder, case_idx):
    db = FRODO(root_dir = datafolder, format = 'CODA', initial_parse = True)
    
    db.extract_inputs(
        id_groups = (3,),
        cases_idx = case_idx,
        vtu_type='surface',
        verbose=False
        )

    # db.extract_inputs(
    #     id_groups = (4,),
    #     cases_idx = case_idx,
    #     vtu_type='volume',
    #     verbose=False
    #     )
    
    for stage in [0, 1]:
        db.extract_outputs(
            id_groups=(3,),
            stage=stage, cases_idx = case_idx,
            var_name_excluded = [
                'BoundaryValues_CoefSkinFrictionX',
                'BoundaryValues_CoefSkinFrictionY',
                'BoundaryValues_CoefSkinFrictionZ'
                ],
            vtu_type='surface',
            )
        
        # db.extract_outputs(
        #     id_groups=(4,),
        #     stage=stage, cases_idx = case_idx,
        #     var_name_excluded = [],
        #     vtu_type='volume',
        #     )
    
    return db

case_idx = list(range(5))
db_0 = read_db(
    datafolder = '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_basic/',
    case_idx = case_idx,
    )

0 Warning! Import - NVTX not present!



 NEW CODA SIMULATION WILL BE LOADED FROM /home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_basic
100 simulations found.
Parse took: 0.0837 s


In [2]:
from FotR import SAM
import matplotlib.pyplot as plt
import imageio
list_fig = []
xyz_sort, order_sort = SAM.Weapons.sort_by_centroid(db_0.data_dict['CADGroup_3']['Coord'])
cp = db_0.data_dict['CADGroup_3']['Vars']['0']['BoundaryValues_CoefPressure'][order_sort, :]

scale=7

case = 2


In [3]:
import numpy as np
diff = np.diff(cp, axis=0)
print(cp.shape, diff.shape, np.max(diff))

(13862, 5) (13861, 5) 0.2151087029985892


In [ ]:

for stencil in range(10, 300):
    grad_cp = SAM.DifferentialOperators.gradient(
        X = xyz_sort,
        f = cp,
        stencil_width=stencil,
        poly_order = 2
    )

    mask_intrados = xyz_sort[:, 2] > 0
    
    fig, ax = plt.subplots(1, 1, figsize=(7, 5))
    ax.scatter(
        xyz_sort[mask_intrados, 0],
        xyz_sort[mask_intrados, 2],
        c='black',
        s=1
    )
    ax.set_xlabel('x')
    ax.set_ylabel('z')
    ax.set_ylim(bottom = xyz_sort[:, 2].min()*scale, top = xyz_sort[:, 2].max()*scale)

    ax_cp = ax.twinx()
    ax_cp.scatter(
        xyz_sort[mask_intrados, 0], cp[mask_intrados, case], c='green', s=2
    )
    ax_cp.spines['left'].set_position(('outward', 60))
    ax_cp.spines['left'].set_color('green')
    ax_cp.tick_params(axis='y', colors='green')
    ax_cp.invert_yaxis()
    ax_cp.yaxis.set_label_position('left')
    ax_cp.yaxis.tick_left()
    ax_cp.spines['right'].set_visible(False)

    ax_twin = ax.twinx()
    ax_twin.scatter(
        xyz_sort[mask_intrados, 0], grad_cp[0, mask_intrados, case], s=3, c='red'
    )
    ax_twin.set_yscale('symlog')
    # ax_twin.spines['right'].set_position(('outward', 30 +_ * 30))
    ax_twin.spines['right'].set_color('red')
    ax_twin.tick_params(axis='y', colors='red')
    fig.suptitle(f'Case {case} - stencil {stencil}')
    list_fig.append(fig)
    plt.close(fig)
    # fig.savefig(f'/home/m.jaraiz/repos/fellowship-of-the-ring/examples/pictures_diff/s_{stencil}.png')

import os
folder_path = '/home/m.jaraiz/repos/fellowship-of-the-ring/examples/pictures_diff'
gif_path = os.path.join(folder_path, f'case_{case}.gif')
with imageio.get_writer(gif_path, mode='I', duration=1.5) as writer:
    for fig in list_fig:
        # Save the figure to a temporary file
        temp_path = os.path.join(folder_path, 'temp.png')
        fig.savefig(temp_path)
        plt.close(fig)
        # Read the image and append it to the gif
        image = imageio.imread(temp_path)
        writer.append_data(image)
        # Remove the temporary file
        os.remove(temp_path)

#### Ejemplo GIMLI

In [ ]:
"""
Ejemplo mínimo de uso de GIMLI sobre un DataFrame sintético.

Simula 3 "casos" CFD (AoA distintos) con un Cp distribuido en dos
"familias" de comportamiento (p.ej. zona laminar / zona con shock),
para comprobar que prepare -> select_model -> fit -> predict -> summary
funciona de punta a punta y que los resultados son razonables.
"""

import numpy as np
import pandas as pd

from FotR import GIMLI

rng = np.random.default_rng(0)

rows = []
for aoa in (1.0, 3.0, 5.0):
    n = 200
    # Dos sub-poblaciones de Cp -> deberíamos recuperar 2 clusters por caso
    cp_a = rng.normal(loc=-0.2, scale=0.05, size=n // 2)
    cp_b = rng.normal(loc=-0.8, scale=0.05, size=n // 2)
    cp = np.concatenate([cp_a, cp_b])
    x = rng.uniform(0, 1, size=n)
    rows.append(pd.DataFrame({"AoA": aoa, "x": x, "Cp": cp}))

df = pd.concat(rows, ignore_index=True)
print("df shape:", df.shape)
print(df.head())

In [ ]:
import matplotlib.pyplot as plt

case = 2 # De los tres casos de aoa

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
df_case = df[df["AoA"].unique()[case] == df["AoA"]]
ax.scatter(df_case["x"], df_case["Cp"], s=5)
ax.set_title(f"Case {case} (AoA={df_case['AoA'].iloc[0]})")


In [ ]:
cluster = GIMLI(
    df,
    variables=["Cp"],
    groups=["AoA"],
    algorithm="gmm",
    cluster_range=range(1, 5),
    covariance_types=("diag",),
    verbose=True,
)

cluster.prepare()
cluster.select_model()
cluster.fit()
cluster.predict()
table = cluster.summary()

cluster.plot.bic(group=(1.0,))       # curva BIC vs n_components para AoA=1.0
cluster.plot.scatter(group=(1.0,))   # scatter coloreado por cluster (cae a 1D + PCA si hiciera falta)

print("\nrepr:", cluster)
print("\nsummary table:\n", table)

df_out = cluster.dataset.dataframe
print("\nclustered head:\n", df_out.head())

assert "cluster" in df_out.columns
assert "cluster_proba_max" in df_out.columns
assert (table["recommended_n"] == 2).all(), "Esperaba recuperar 2 clusters por caso"


print("\nOK: ejemplo ejecutado correctamente.")